# Per-fold F-max scoring (full-pool and per-aspect splits)

Computes weighted F-max for each base model × each CV fold × each GO aspect
(MFO / BPO / CCO), using the precomputed outputs from
`src/create_ensemble_datasets.py`:

- `train_merged.tsv` — one confidence column per base model
- `train_terms.tsv`  — ground-truth GO annotations
- `IA.txt`           — information-accretion weights
- `splits/f{k}_split.csv`                 — full-pool fold assignments
- `splits/f{k}_split_{mf,bp,cc}.csv`      — per-aspect fold assignments

We run **two** evaluations:

1. **full_pool** — one split per fold, shared across aspects. `fmax` is called
   once per (model, fold) on all-aspect ground truth; the returned dict gives
   all three aspects.
2. **per_aspect** — separate split per (aspect, fold). `fmax` is called once
   per (aspect, fold, model) and only the relevant aspect is kept.

Result is written to `data/final_data/fmax_per_fold.json` as
`{pool: {model: {aspect: [fmax_fold0, ..., fmax_foldK-1]}}}`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.metrics import fmax, ASPECTS

In [ ]:
# ---- config ----
DATA_DIR    = Path('data/final_data')
SPLITS_DIR  = DATA_DIR / 'splits'
N_FOLDS     = 5
OUTPUT_JSON = DATA_DIR / 'fmax_per_fold.json'

# aspect code in train_terms.tsv -> suffix used in per-aspect split filenames
ASPECT_SUFFIX = {'MFO': 'mf', 'BPO': 'bp', 'CCO': 'cc'}

In [ ]:
# ---- load precomputed data ----
train_merged = pd.read_csv(DATA_DIR / 'train_merged.tsv', sep='\t')
print(f"train_merged: {len(train_merged):>10,} rows, "
      f"{train_merged['protein_id'].nunique():,} proteins")

train_terms = pd.read_csv(DATA_DIR / 'train_terms.tsv', sep='\t')
print(f"train_terms:  {len(train_terms):>10,} annotations, "
      f"{train_terms['EntryID'].nunique():,} proteins")

# IA.txt is tab-separated: term\tia
ia_weights = {}
with open(DATA_DIR / 'IA.txt') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            ia_weights[parts[0]] = float(parts[1])
print(f"ia_weights:   {len(ia_weights):>10,} terms")

# model names come from the conf_<n> columns
model_cols  = [c for c in train_merged.columns if c.startswith('conf_')]
model_names = [c[len('conf_'):] for c in model_cols]
print(f"models:       {model_names}")

In [ ]:
# ---- helper: slice this model's predictions to a val-protein frame ----
def model_preds_for(val_preds, model):
    """Return (protein_id, GO_term, confidence) for a single model; drop zeros.

    Filtering zero confidences is safe because the threshold sweep in fmax
    starts at 0.01, and it meaningfully shrinks the per-protein dicts fmax
    builds internally.
    """
    preds = (
        val_preds[['protein_id', 'GO_term', f'conf_{model}']]
        .rename(columns={f'conf_{model}': 'confidence'})
    )
    return preds[preds['confidence'] > 0]


# pre-index ground truth by aspect (used by the per-aspect loop below)
gt_by_aspect = {a: train_terms[train_terms['aspect'] == a] for a in ASPECTS}
for a, df in gt_by_aspect.items():
    print(f"  {a}: {len(df):>10,} annotations, {df['EntryID'].nunique():,} proteins")

In [ ]:
# =============================================================================
# Evaluation 1: full-pool splits (f{k}_split.csv)
#   Same val proteins for all aspects -> fmax reports all three in one call.
# =============================================================================
results_full = {m: {a: [] for a in ASPECTS} for m in model_names}

for fold in range(N_FOLDS):
    split_path = SPLITS_DIR / f'f{fold}_split.csv'
    if not split_path.exists():
        print(f"fold {fold}: {split_path} missing — skipping")
        continue

    split        = pd.read_csv(split_path)
    val_proteins = set(split.loc[split['split'] == 'val', 'protein_id'])
    val_preds    = train_merged[train_merged['protein_id'].isin(val_proteins)]
    val_gt       = train_terms[train_terms['EntryID'].isin(val_proteins)]
    print(f"\nfold {fold}: {len(val_proteins):,} val proteins, "
          f"{len(val_gt):,} GT annotations, {len(val_preds):,} pred rows")

    for model in model_names:
        preds  = model_preds_for(val_preds, model)
        scores = fmax(preds, val_gt, ia_weights)
        for aspect in ASPECTS:
            results_full[model][aspect].append(float(scores[aspect]))
        aspect_str = "  ".join(f"{a}={float(scores[a]):.4f}" for a in ASPECTS)
        print(f"    {model:<28} {aspect_str}")

In [ ]:
# =============================================================================
# Evaluation 2: per-aspect splits (f{k}_split_{mf,bp,cc}.csv)
#   Val proteins differ across aspects -> iterate (aspect, fold, model).
# =============================================================================
results_per = {m: {a: [] for a in ASPECTS} for m in model_names}

for aspect in ASPECTS:
    suffix    = ASPECT_SUFFIX[aspect]
    aspect_gt = gt_by_aspect[aspect]
    print(f"\n=== {aspect} (suffix=_{suffix}) ===")

    for fold in range(N_FOLDS):
        split_path = SPLITS_DIR / f'f{fold}_split_{suffix}.csv'
        if not split_path.exists():
            print(f"  fold {fold}: {split_path} missing — skipping")
            continue

        split        = pd.read_csv(split_path)
        val_proteins = set(split.loc[split['split'] == 'val', 'protein_id'])
        val_preds    = train_merged[train_merged['protein_id'].isin(val_proteins)]
        val_gt       = aspect_gt[aspect_gt['EntryID'].isin(val_proteins)]
        print(f"  fold {fold}: {len(val_proteins):,} val proteins, "
              f"{len(val_gt):,} GT annotations, {len(val_preds):,} pred rows")

        for model in model_names:
            preds  = model_preds_for(val_preds, model)
            scores = fmax(preds, val_gt, ia_weights)
            f_asp  = float(scores[aspect])
            results_per[model][aspect].append(f_asp)
            print(f"      {model:<28} fmax = {f_asp:.4f}")

In [ ]:
# ---- save both pools to one JSON ----
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
payload = {
    'full_pool':  results_full,
    'per_aspect': results_per,
}
with open(OUTPUT_JSON, 'w') as f:
    json.dump(payload, f, indent=2)
print(f"Saved -> {OUTPUT_JSON}")

In [ ]:
# ---- summary: mean ± std across folds, per pool per model per aspect ----
def print_summary(results, title):
    header = f"{'model':<28}" + "".join(f"{a:>18}" for a in ASPECTS) + f"{'overall':>12}"
    print(f"\n{title}")
    print('=' * len(header))
    print(header)
    print('-' * len(header))
    for model in model_names:
        per_aspect_means = []
        row = f"{model:<28}"
        for aspect in ASPECTS:
            vals = np.array(results[model][aspect])
            if len(vals) == 0:
                row += f"{'-':>18}"
                continue
            row += f"{vals.mean():.4f}±{vals.std():.4f}".rjust(18)
            per_aspect_means.append(vals.mean())
        row += (f"{np.mean(per_aspect_means):.4f}".rjust(12)
                if per_aspect_means else f"{'-':>12}")
        print(row)

print_summary(results_full, 'FULL POOL SPLITS')
print_summary(results_per,  'PER-ASPECT SPLITS')